# Where the forecasts fail: a post-mortem of the worst days

The backtest gives one number per model. Someone who trades on a forecast asks a different question: on which days would it have been badly wrong, and why? A model's average error says little about that, because power price errors are not spread evenly. Most days are ordinary and easy; a few days carry a large share of the damage.

This notebook looks at the gradient-boosted trees (`gbm`), with the linear model and the weekly naive benchmark (`naive_weekly`) alongside so each failure can be judged against the others. It has three steps:

- **Step 2** asks which kinds of hours the error comes from, grouped by how high or low the actual price was.
- **Step 3** lists the ten delivery days with the largest error and draws each one hour by hour.
- **Step 4** takes the single worst day apart, with the inputs the model was given.

The last section is for you to write: what the failures have in common, in your own words. That section is what an interviewer will ask about.

### Before running

The notebook reads two tables and writes nothing to the database. `predictions` must hold the `naive_weekly`, `linear` and `gbm` backtests (run `uv run python -m pl_dayahead.backtest naive_weekly linear gbm` if one is missing). `hourly` provides the load, wind and solar forecasts. Figures are saved to `reports/figures/`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pl_dayahead import store
from pl_dayahead.config import PROJECT_ROOT
from pl_dayahead.holidays_pl import is_holiday
from pl_dayahead.timeutils import LOCAL_TZ

# Chart styling, the same as notebook 01. Colours come from a palette tested for
# colour-blind readability.
INK, INK_2, MUTED = '#0b0b0b', '#52514e', '#898781'  # text: primary, secondary, muted
GRID, BASELINE, SURFACE = '#e1e0d9', '#c3c2b7', '#fcfcfb'  # gridlines, axis line, background

# Each colour belongs to one thing in every chart of this notebook: the actual price is
# always ink, gbm always blue, the linear model always orange. The inputs get their own.
SERIES = {'actual': INK, 'gbm': '#2a78d6', 'linear': '#eb6834'}
INPUTS = {'load': INK_2, 'wind': '#1baf7a', 'solar': '#eda100'}

plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'font.family': 'sans-serif', 'font.size': 10,
    'text.color': INK, 'axes.labelcolor': INK_2, 'axes.titlecolor': INK,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left', 'axes.titlepad': 12,
    'xtick.color': MUTED, 'ytick.color': MUTED, 'xtick.labelcolor': INK_2, 'ytick.labelcolor': INK_2,
    'axes.edgecolor': BASELINE, 'axes.linewidth': 0.8,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.spines.left': False,
    'axes.grid': True, 'axes.grid.axis': 'y', 'axes.axisbelow': True,
    'grid.color': GRID, 'grid.linewidth': 0.6,
    'legend.frameon': False,
})
FIG_DIR = PROJECT_ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
GW = 1000  # the forecasts are stored in MW; charts and tables show GW

## Step 1: load the forecasts and the inputs

The `predictions` table has one row per model per hour: the forecast, the actual price, and when the model was trained. It is reshaped so that each hour is one row with a column per model, then joined to the `hourly` table so every hour also carries the load, wind and solar forecasts the models were given.

Only hours that every model forecast are kept, exactly as `evaluate.py` does, so the MAE printed at the end of the cell must match the Overall table from `uv run python -m pl_dayahead.evaluate`. If it does not, something in this notebook is wrong, not in the models.

The error of a forecast is defined as **forecast minus actual**. A positive error means the forecast was too high, a negative one that it was too low. The sign matters below: the size of the error says how bad a miss was, the sign says which way the model leans.

In [ ]:
con = store.connect()
predictions = con.sql('SELECT model, ts_utc, prediction, actual FROM predictions').df()
staged = con.sql('SELECT ts_utc, load_fcst, wind_fcst, solar_fcst, residual_load_fcst FROM hourly').df()
con.close()

MODELS = ['gbm', 'linear', 'naive_weekly']

# One row per hour, one forecast column per model, plus the actual price.
wide = predictions.pivot_table(index='ts_utc', columns='model', values='prediction', aggfunc='first')[MODELS]
wide['actual'] = predictions.groupby('ts_utc')['actual'].first()
wide = wide.dropna().reset_index()  # keep only hours every model forecast
wide = wide.merge(staged, on='ts_utc', how='left')

# Warsaw time, delivery day and hour of day, as in notebook 01.
wide['local'] = wide['ts_utc'].dt.tz_convert(LOCAL_TZ)
wide['day'] = pd.to_datetime(wide['local'].dt.date)
wide['hour'] = wide['local'].dt.hour

for m in MODELS:
    wide[f'err_{m}'] = wide[m] - wide['actual']  # positive: forecast too high
    wide[f'abs_{m}'] = wide[f'err_{m}'].abs()

print(f"{len(wide):,} hours on {wide['day'].nunique():,} delivery days, "
      f"{wide['day'].min():%Y-%m-%d} to {wide['day'].max():%Y-%m-%d}")
print('MAE, EUR/MWh (must match evaluate.py):', {m: round(wide[f'abs_{m}'].mean(), 2) for m in MODELS})

## Step 2: which hours carry the error?

MAE averages over all hours equally, which hides where the error actually sits. Here every hour is put in a band by its **actual** price, and for each band the table shows:

- **share of hours %**: how common prices in that band are;
- **share of gbm error %**: how much of gbm's total absolute error those hours account for. If a band has 3% of the hours but 15% of the error, its hours are five times harder than average;
- **MAE** of gbm and the linear model within the band, to see where the trees gain;
- **mean error** (the average of forecast minus actual, with its sign): close to zero means the model misses in both directions equally; far from zero means it leans one way in that band.

The bands are fixed amounts in EUR/MWh rather than percentiles, so they read directly: "below 0" is every negative-price hour, "250 and above" the spikes.

One pattern to expect. A model trained to minimise absolute error learns to forecast the typical outcome for a given situation, and most training hours are ordinary. So in the extreme bands it is pulled toward ordinary prices: too high when the price goes negative (positive mean error) and too low when it spikes (negative mean error). Every statistical model does this to some degree; the question is how much.

In [ ]:
BANDS = [-np.inf, 0, 50, 100, 150, 250, np.inf]
LABELS = ['below 0', '0 to 50', '50 to 100', '100 to 150', '150 to 250', '250 and above']
# right=False makes each band include its lower edge: 0 goes in '0 to 50', 250 in '250 and above'.
wide['band'] = pd.cut(wide['actual'], BANDS, labels=LABELS, right=False)

by_band = wide.groupby('band', observed=False)
regimes = pd.DataFrame({
    'hours': by_band.size(),
    'share of hours %': 100 * by_band.size() / len(wide),
    'share of gbm error %': 100 * by_band['abs_gbm'].sum() / wide['abs_gbm'].sum(),
    'MAE gbm': by_band['abs_gbm'].mean(),
    'MAE linear': by_band['abs_linear'].mean(),
    'mean error gbm': by_band['err_gbm'].mean(),
    'mean error linear': by_band['err_linear'].mean(),
})
regimes.round(1)

In [ ]:
# Chart 1: share of hours against share of gbm's error, band by band.
fig, ax = plt.subplots(figsize=(8, 4.2))
y = np.arange(len(LABELS))
h = 0.36  # bar thickness; the two bars of a band sit just above and below its centre line
hours_bars = ax.barh(y - h / 2 - 0.01, regimes['share of hours %'], height=h, color=BASELINE, label='share of hours')
error_bars = ax.barh(y + h / 2 + 0.01, regimes['share of gbm error %'], height=h, color=SERIES['gbm'],
                     label='share of gbm error')

# Write each value at the end of its bar, in text colour (never the bar colour).
for bars in (hours_bars, error_bars):
    for bar in bars:
        value = bar.get_width()
        text = f'{value:.1f}%' if value < 10 else f'{value:.0f}%'  # a decimal for small shares, so 0.3% is not shown as 0%
        ax.text(value + 0.4, bar.get_y() + bar.get_height() / 2, text, va='center', fontsize=9, color=INK_2)

ax.set_yticks(y, LABELS)
ax.invert_yaxis()  # lowest prices at the top, reading downward like the table
ax.grid(axis='y', visible=False)
ax.grid(axis='x', visible=True)
ax.set_xlabel('% of all hours, % of total absolute error')
ax.set_ylabel('actual price, EUR/MWh')
ax.set_title('Where the error comes from', pad=22)
ax.text(0, 1.02, 'gbm forecast, all hours 2024 to 2026, grouped by the actual price',
        transform=ax.transAxes, color=INK_2, fontsize=9)
ax.legend(loc='lower right')
fig.tight_layout()
fig.savefig(FIG_DIR / 'error_by_price_level.png', dpi=150)
plt.show()

### What to look for

Compare the two bars in each band. Where the blue bar is much longer than the grey one, those hours carry far more than their share of the error; those are the hours a better model would have to fix.

Then read the mean error column of the table. A positive value in "below 0" and a negative one in "250 and above" is the pull toward ordinary prices described above. Its size tells you how far off the model typically is on the days that matter most to a trader.

Finally compare MAE gbm with MAE linear band by band. Whichever model wins overall, this shows whether it wins by handling the extremes better or by being more precise on ordinary hours.

## Step 3: the ten worst days

The auction sets all of a day's prices at once, and a trader's position covers the whole day, so the day is the natural unit for a post-mortem. Each delivery day gets gbm's MAE over its hours, and the ten largest are listed.

The columns, all in EUR/MWh unless marked GW:

- **mae_gbm, mae_linear, mae_naive**: each model's average miss that day (mae_naive is the weekly naive). If all three are large, the day could not be forecast from these inputs; if only gbm is large, it is a weakness of the trees.
- **bias_gbm**: gbm's mean error that day. Negative means it forecast too low across the day.
- **actual_min, actual_max** against **gbm_min, gbm_max**: the range of the day's prices and of the forecast. A spike shows as actual_max far above gbm_max, a price collapse as actual_min far below gbm_min.
- **jump**: the day's average price minus the previous day's. The previous day's prices are the models' strongest input, so a large jump means that input pointed the wrong way.
- **wind_gw, solar_gw, load_gw**: the day's average forecast of each input, to see what kind of day it was.

Three kinds of miss tend to show up. A **spike**: an evening with little wind and high demand, when the most expensive plants set the price. A **collapse**: a sunny or windy weekend or holiday with negative prices. A **level jump**: the whole day priced far from the day before, often after a change in fuel prices or a large power plant going offline, neither of which the models are given.

In [ ]:
daily = wide.groupby('day').agg(
    mae_gbm=('abs_gbm', 'mean'), mae_linear=('abs_linear', 'mean'), mae_naive=('abs_naive_weekly', 'mean'),
    bias_gbm=('err_gbm', 'mean'),
    actual_mean=('actual', 'mean'), actual_min=('actual', 'min'), actual_max=('actual', 'max'),
    gbm_min=('gbm', 'min'), gbm_max=('gbm', 'max'),
    wind_gw=('wind_fcst', 'mean'), solar_gw=('solar_fcst', 'mean'), load_gw=('load_fcst', 'mean'),
)
daily[['wind_gw', 'solar_gw', 'load_gw']] /= GW

# The previous calendar day's average price, looked up by date so a missing day can never
# make a day compare itself with the wrong neighbour.
previous = daily['actual_mean'].reindex(daily.index - pd.Timedelta(days=1)).to_numpy()
daily['jump'] = daily['actual_mean'] - previous

daily['weekday'] = daily.index.day_name().str[:3]
daily['holiday'] = is_holiday(daily.index.to_series()).to_numpy()

worst = daily.sort_values('mae_gbm', ascending=False).head(10)
COLUMNS = ['weekday', 'holiday', 'mae_gbm', 'mae_linear', 'mae_naive', 'bias_gbm',
           'actual_min', 'actual_max', 'gbm_min', 'gbm_max', 'jump', 'wind_gw', 'solar_gw', 'load_gw']
worst[COLUMNS].round(1)

In [ ]:
# Chart 2: the ten worst days, hour by hour. Each panel has its own price scale, because
# the days differ so much that a shared scale would flatten most of them.
fig, axes = plt.subplots(2, 5, figsize=(16, 7), sharex=True)
for ax, (day, row) in zip(axes.flat, worst.iterrows()):
    d = wide[wide['day'] == day].sort_values('ts_utc')
    ax.axhline(0, color=BASELINE, lw=0.8)
    ax.plot(d['hour'], d['actual'], color=SERIES['actual'], lw=2, label='actual price')
    ax.plot(d['hour'], d['gbm'], color=SERIES['gbm'], lw=2, label='gbm forecast')
    ax.plot(d['hour'], d['linear'], color=SERIES['linear'], lw=1.5, label='linear forecast')
    ax.set_title(f"{day:%a %d %b %Y}", fontsize=10, pad=18)
    ax.text(0, 1.02, f"gbm MAE {row['mae_gbm']:.0f} EUR/MWh", transform=ax.transAxes, fontsize=8.5, color=INK_2,
            va='bottom', ha='left')
    ax.set_xticks([0, 6, 12, 18, 23])
for ax in axes[:, 0]:
    ax.set_ylabel('EUR/MWh')
for ax in axes[1]:
    ax.set_xlabel('hour, Warsaw time')

handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper left', ncol=3, bbox_to_anchor=(0.01, 0.93))
fig.text(0.01, 0.975, 'The ten days with the largest gbm error', fontsize=13, fontweight='bold', va='top')
fig.tight_layout(rect=(0, 0, 1, 0.88))
fig.savefig(FIG_DIR / 'worst_days.png', dpi=150)
plt.show()

## Step 4: the worst day in detail

The top chart is the day's prices; the bottom chart is the day-ahead forecasts of load, wind and solar that the models were given, in GW. They share the hour axis, so a miss in the top chart can be read straight down to what the inputs were doing at that hour. They are two charts rather than one with two y-axes on purpose: two scales on one chart invite comparing heights that mean nothing.

To look at another day, set `DAY` to its date, for example `DAY = pd.Timestamp('2025-06-08')`.

One limit of this view: the inputs are **forecasts**. If the wind blew less than forecast, the price rose for a reason the model could not see, and this chart cannot tell that apart from a model that misread a correct forecast. Separating the two needs the actual wind and solar output, which ENTSO-E also publishes ("actual generation per production type") and which is the natural next dataset to add.

In [ ]:
DAY = worst.index[0]  # the worst day; replace with any date to inspect it
d = wide[wide['day'] == DAY].sort_values('ts_utc')


def label_ends(ax, items, min_gap=0.07):
    # Write each series name just right of its last point, in text colour, nudging labels
    # apart when the lines end close together. items: (name, last y value).
    ymin, ymax = ax.get_ylim()
    placed = []
    for name, y_end in sorted(items, key=lambda item: item[1]):
        pos = (y_end - ymin) / (ymax - ymin)  # position as a fraction of the axis height
        if placed and pos - placed[-1] < min_gap:
            pos = placed[-1] + min_gap
        placed.append(pos)
        ax.text(1.01, pos, name, transform=ax.transAxes, va='center', fontsize=9, color=INK_2)


fig, (top, bottom) = plt.subplots(2, 1, figsize=(10, 7.5), sharex=True, gridspec_kw={'height_ratios': [3, 2]})

top.axhline(0, color=BASELINE, lw=0.8)
for name, width in [('actual', 2), ('gbm', 2), ('linear', 1.5)]:
    top.plot(d['hour'], d[name], color=SERIES[name], lw=width, label=name)
top.set_ylabel('EUR/MWh')
top.legend(loc='best', ncol=3)
label_ends(top, [(name, d[name].iloc[-1]) for name in ['actual', 'gbm', 'linear']])

for name, column in [('load', 'load_fcst'), ('wind', 'wind_fcst'), ('solar', 'solar_fcst')]:
    bottom.plot(d['hour'], d[column] / GW, color=INPUTS[name], lw=2, label=name)
bottom.set_ylim(0, d[['load_fcst', 'wind_fcst', 'solar_fcst']].max().max() / GW * 1.3)  # headroom for the legend
bottom.set_ylabel('GW, day-ahead forecast')
bottom.set_xlabel('hour, Warsaw time')
bottom.set_xticks(range(0, 24, 3))
bottom.legend(loc='best', ncol=3)
label_ends(bottom, [(name, d[column].iloc[-1] / GW) for name, column in
                    [('load', 'load_fcst'), ('wind', 'wind_fcst'), ('solar', 'solar_fcst')]])

row = daily.loc[DAY]
fig.text(0.01, 0.985, f"{DAY:%A %d %B %Y}: prices and the inputs the models were given", fontsize=13,
         fontweight='bold', va='top')
fig.text(0.01, 0.95, f"MAE gbm {row['mae_gbm']:.0f}, linear {row['mae_linear']:.0f}, naive {row['mae_naive']:.0f} "
         f"EUR/MWh; day average {row['actual_mean']:.0f}, {row['jump']:+.0f} on the day before",
         fontsize=9.5, color=INK_2, va='top')
fig.tight_layout(rect=(0, 0, 0.95, 0.93))
fig.savefig(FIG_DIR / 'worst_day_detail.png', dpi=150)
plt.show()

## What I see

Write this section yourself once the notebook has run; it is the part of the project an interviewer will probe, so it should be in your words. Four questions to answer, each in a sentence or two with the number that supports it:

1. **Where does the error sit?** What share of gbm's error comes from the "below 0" and "250 and above" bands, and what share of the hours do they make up?
2. **Which way does the model lean?** What are the mean errors in those two bands, and what does that say about trusting the forecast on extreme days?
3. **What kind of days are the ten worst?** Count how many are spikes, collapses and level jumps, and on how many all three models failed together.
4. **What would have helped on the worst day?** Name the information the models did not have that a trader would have looked at, such as plant outages, gas prices or actual wind output.

## Limits of this analysis

- The linear model and the trees here are the main versions. They use the renewables forecast for the delivery day, published at 18:00 the day before, after the 11:00 issue time, as the papers do (see `availability.py`). The comparison versions (`linear_res_d1`, `gbm_res_d1`) use the previous day's forecast instead; `evaluate.py` reports how much the main versions gain from it.
- The models are not given fuel or carbon prices, cross-border flows or power plant outages. Some of the worst days are likely explained by one of these, and this notebook can only point to them, not prove it.
- Ten days is a small sample. They show the kinds of failure, not how often each happens; Step 2 answers the "how often" question for hours.